# Phase 11A — Historical enhancer test overlap and its measured effect

This notebook fits three small sequence baselines under three **prespecified training cohorts**, predicts the **same supplied historical independent test** in each cohort, and compares the full Stage 1 test with its nonduplicated subset. It also records the already frozen Phase 2.1 similarity-grouped CV estimates as **separate context**. It does not rerun or overwrite Phases 1–10.

**Kaggle Inputs (three saved datasets; attach their ZIPs or Kaggle-unpacked contents):**

1. `DNABERT (Preserve).zip` — the eight original `Dataset/*.txt` FASTA inputs.
2. `[1] enhancer_audit_v1_outputs.zip` — Stage 1/2 candidate manifests and audit summary.
3. `[2.1] enhancer_phase2_corrected_results.zip` — corrected grouped-CV metrics and correction report.

Kaggle CPU is sufficient; Internet and GPU can be off. Run all cells and download `enhancer_phase11a_overlap_impact.zip` from notebook Output. The ZIP holds the test predictions, per-cohort metrics, paired intervals, overlap roster, sham-removal controls, grouped-CV context, provenance, and figures.

**Interpretation guardrail:** In these supplied files all 100 Stage 2 positive historical test entries also occur in training. The remaining Stage 2 test has only negatives; a cleaned Stage 2 AUROC/AUPRC is **undefined**. The historical test was previously examined, and the provenance of its contradiction with a publisher nonoverlap claim is unresolved. Report an effect in these **supplied files**, not a universal correction of every published experiment.

In [ ]:
# 0 — Imports, exact input-member locks, protocol and workspace.
import os, io, re, json, zipfile, hashlib, itertools, warnings, platform
from pathlib import Path
import numpy as np, pandas as pd
import scipy, sklearn
from scipy import sparse
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.preprocessing import StandardScaler, Normalizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
ROOT=Path('/kaggle/input') if Path('/kaggle/input').exists() else Path('upload')
WORK=Path('/kaggle/working/enhancer_phase11a_work') if Path('/kaggle/working').exists() else Path('phase11a_work')
WORK.mkdir(parents=True,exist_ok=True)
CACHE=WORK/'fit_cache';CACHE.mkdir(exist_ok=True)
HASH={
 'enhancer.cv.txt':'5aa1336bc38a675f579244e9dabb3c689abb835880eb42b1ba54451cc667ba61',
 'enhancer.ind.txt':'acaea2aca7113947b47d6b672848d07be9cf3cacd7a0c2cf84f554d197cda374',
 'non.cv.txt':'661abf8105ff64905d9a64b609c4112af637ee44b3157ad8742f03960ded3412',
 'non.ind.txt':'deb9f732de90814b0b80bf48e224778746765eb8915a5c0f4501de8510eec3fa',
 'strong_742.txt':'1e19da34bec49d5425faaa1841feb6086e2c306f20f1ffca54e262f61e058dcb',
 'strong_100.txt':'0b808bfa688475c65b0f61aa5f4bb38518369501cb518f6fe0a18093957a250d',
 'weak_742.txt':'c9a74473bbf5001e3ef123c889bffe814b610baa15b80d18b1112b07ca857bf9',
 'weak_100.txt':'0d8a382444797696afb45f1484db9aa5393acaac11f931eb7f0b9f292871a6b9',
 'audit_run_summary.json':None,
 'stage1_candidate_manifest.csv':'e295155300fafc34528bc1868c6fad6ec78f3912c5b07e034710e81c3b4ac4b3',
 'stage2_candidate_manifest.csv':'e25291665c8c42522b662855d3774a0e425e37f603c2ab1f22f4a2c241def952',
 'correction_report.json':None,
 'comparison_mean_fold_vs_pooled.csv':'4900d1eb48ce357fa8712476a625b8d46a2a66aa5a89f9de8d289e16db886b74'}
def sha(b):return hashlib.sha256(b).hexdigest()
CONFIG={'name':'phase11a_supplied_historical_overlap_impact_v1',
 'input_member_hashes':{k:v for k,v in HASH.items() if v},
 'training_cohorts':['historical_raw','remove_exact_test_matches','remove_related_test_groups'],
 'models':['gc_cpg_logreg','k3_logreg','spaced6_4_logreg'],
 'rc_augment_train':True,'logistic_C':1.0,'max_iter':600,
 'sham_repeats':5,'bootstrap_repeats':1000,'seed':20261015,
 'historical_test_status':'secondary; previously examined',
 'stage2_clean_discrimination':'undefined after all positive test matches removed'}
LOCK=sha(json.dumps(CONFIG,sort_keys=True,separators=(',',':')).encode())
print('Protocol',LOCK,'| root',ROOT,'| output',WORK)
assert ROOT.exists(), 'Attach the three saved Kaggle Inputs listed above.'

In [ ]:
# 1 — Locate by authenticated member bytes, including Kaggle-unpacked ZIPs.
# Restrict ZIP scanning to relevant uploaded archives; accept arbitrary Kaggle dataset folder names.
ZIP_NAMES={'DNABERT (Preserve).zip','[1] enhancer_audit_v1_outputs.zip',
 '[2.1] enhancer_phase2_corrected_results.zip'}
ZIP_FILES=[p for p in ROOT.rglob('*.zip') if p.name in ZIP_NAMES]
ZIP_FILES+= [p for p in ROOT.rglob('*.zip') if p.name not in ZIP_NAMES and any(
 t in p.name.lower() for t in ('audit_v1','phase2_corrected','dnabert'))]
ZIP_FILES=sorted(set(ZIP_FILES))
def inputs_for(name):
 hits=[];bad=[]
 for p in ROOT.rglob(name):
  if p.is_file():
   raw=p.read_bytes()
   (hits if HASH[name] is None or sha(raw)==HASH[name] else bad).append((raw,str(p)))
 for p in ZIP_FILES:
  try:
   with zipfile.ZipFile(p) as z:
    matches=[m for m in z.namelist() if m==name or m.endswith('/'+name)]
    for member in matches:
     raw=z.read(member)
     (hits if HASH[name] is None or sha(raw)==HASH[name] else bad).append((raw,str(p)+'!'+member))
  except (OSError,zipfile.BadZipFile): continue
 if not hits:raise FileNotFoundError(f'Missing or wrong-version {name}; unmatched candidates: {[x[1] for x in bad[:8]]}')
 if len({sha(x[0]) for x in hits})!=1:raise ValueError('Ambiguous versions of '+name)
 print('VERIFIED',name,'via',hits[0][1])
 return hits[0][0]
# Manifest-derived expected hashes authenticate their own audit summaries.
summary=json.loads(inputs_for('audit_run_summary.json'))
for stage in (1,2):
 n=f'stage{stage}_candidate_manifest.csv'
 assert summary['outputs_sha256'][n]==HASH[n]
correction=json.loads(inputs_for('correction_report.json'))
assert correction['output_sha256']['comparison_mean_fold_vs_pooled.csv']==HASH['comparison_mean_fold_vs_pooled.csv']
assert correction['input_manifest_sha256']=={str(s):HASH[f'stage{s}_candidate_manifest.csv'] for s in (1,2)} or correction['input_manifest_sha256']=={f'stage{s}_candidate_manifest.csv':HASH[f'stage{s}_candidate_manifest.csv'] for s in (1,2)}, 'Phase 2.1 manifest lock mismatch'
manifest={s:pd.read_csv(io.BytesIO(inputs_for(f'stage{s}_candidate_manifest.csv')),
 dtype=str,keep_default_na=False) for s in (1,2)}
context=pd.read_csv(io.BytesIO(inputs_for('comparison_mean_fold_vs_pooled.csv')))
RAW={n:inputs_for(n) for n in HASH if n.endswith('.txt')}
for s,d in manifest.items():
 assert len(d)=={1:3268,2:1584}[s] and d.record_id.is_unique
 assert all(sha(x.encode())==h for x,h in zip(d.sequence,d.sequence_sha256))
 assert d.sequence.str.fullmatch('[ACGT]{200}').all()
print('All source members authenticated.')

In [ ]:
# 2 — Parse original FASTAs and audit exact overlap against the locked manifests.
def fasta(raw,source,stage,label,split):
 rows=[];header=None;seq=[]
 for line in raw.decode('utf-8-sig').splitlines()+['>END']:
  line=line.strip()
  if not line:continue
  if line.startswith('>'):
   if header is not None:
    dna=''.join(seq).upper()
    assert re.fullmatch('[ACGT]{200}',dna),(source,header,len(dna))
    rows.append((sha(dna.encode())[:16],dna,label,split,source,header))
   header=line[1:];seq=[]
  else:seq.append(line)
 d=pd.DataFrame(rows,columns=['record_id','sequence','label','original_split','source_file','header'])
 d['stage']=stage
 assert d.record_id.is_unique,(source,'duplicate FASTA entry')
 return d
FILES={1:[('enhancer.cv.txt',1,'train'),('non.cv.txt',0,'train'),
          ('enhancer.ind.txt',1,'test'),('non.ind.txt',0,'test')],
       2:[('strong_742.txt',1,'train'),('weak_742.txt',0,'train'),
          ('strong_100.txt',1,'test'),('weak_100.txt',0,'test')]}
FRAME={}
for s,entries in FILES.items():
 d=pd.concat([fasta(RAW[n],n,s,y,split) for n,y,split in entries],ignore_index=True)
 expected={1:(2968,400),2:(1484,200)}[s]
 assert tuple(d.original_split.value_counts()[['train','test']])==expected
 tr=d[d.original_split=='train'];te=d[d.original_split=='test']
 assert tr.record_id.is_unique and te.record_id.is_unique
 assert not tr.groupby('record_id').label.nunique().gt(1).any()
 overlap=set(tr.record_id)&set(te.record_id)
 assert len(overlap)==100 and set(te[te.record_id.isin(overlap)].label)=={1}
 assert (te[te.record_id.isin(overlap)].label==1).sum()==100
 # Supplied manifests give historical test priority for the 100 shared records.
 m=manifest[s]
 assert set(m.record_id)==set(d.record_id)
 assert set(m.loc[m.role=='legacy_test','record_id'])==set(te.record_id)
 assert set(m.loc[m.role=='train_candidate','record_id']).issubset(set(tr.record_id))
 FRAME[s]=d
 print(f'Stage {s}: original train {len(tr)}, test {len(te)}, exact overlap {len(overlap)}; cleaned test labels',te[~te.record_id.isin(overlap)].label.value_counts().to_dict())
 assert {1:2838,2:1376}[s]==sum(m.role=='train_candidate')
# Record the exact contaminated test entries without exposing any genomic coordinate claim.
roster=pd.concat([FRAME[s].query("original_split == 'test'").assign(
 exact_training_match=lambda x:x.record_id.isin(set(FRAME[s].query("original_split == 'train'").record_id)),
 similarity_group_id=lambda x:x.record_id.map(manifest[s].set_index('record_id').similarity_group_id))
 for s in (1,2)],ignore_index=True)
roster.to_csv(WORK/'historical_test_overlap_roster.csv',index=False)
print('Stage 2 nonduplicate test has one class:',roster.query('stage == 2 and not exact_training_match').label.unique())

## Fixed cohorts and controls

The same historical test is scored every time. `historical_raw` uses the supplied training FASTAs, `remove_exact_test_matches` removes training sequences found verbatim in the test, and `remove_related_test_groups` uses the Phase 1 candidate `train_candidate` role (LCS graph rule). Thus only training changes across these three columns. A second Stage 1 comparison removes duplicated test rows at evaluation time, using each already fitted model without retraining. These two changes answer different questions.

Five prespecified sham controls for the spaced baseline remove the **same number of positive training rows** that are not exact test matches, approximately matched on GC count. They contextualize the effect of deleting positive training rows; they do not prove that every score change was caused solely by leakage. No model selection or threshold tuning uses the historical test.

In [ ]:
# 3 — Fixed feature encodings; cache matrices in memory, fit scalers/IDF within each training cohort.
from itertools import product,combinations
RC_MAP=str.maketrans('ACGT','TGCA')
def rc(s):return s.translate(RC_MAP)[::-1]
UNIVERSE=sorted({s for s in pd.concat([FRAME[1],FRAME[2]]).sequence} |
                {rc(s) for s in pd.concat([FRAME[1],FRAME[2]]).sequence})
INDEX={s:i for i,s in enumerate(UNIVERSE)}
MASKS=list(combinations(range(6),4))
VOC3={''.join(k):i for i,k in enumerate(product('ACGT',repeat=3))}
VOC6={f'{j}'+''.join(k):j*256+i for j in range(len(MASKS)) for i,k in enumerate(product('ACGT',repeat=4))}
def t3(s):return [s[i:i+3] for i in range(198)]
def t6(s):return [f'{j}'+''.join(s[i+k] for k in mask)
 for i in range(195) for j,mask in enumerate(MASKS)]
print('Encoding',len(UNIVERSE),'unique original/RC sequences once. This CPU step can take several minutes.',flush=True)
GC=np.asarray([[sum(s.count(x) for x in 'GC')/200,s.count('CG')/199] for s in UNIVERSE],dtype=np.float32)
X3=CountVectorizer(analyzer=t3,vocabulary=VOC3,dtype=np.float32).transform(UNIVERSE)
X3=Normalizer(norm='l1').fit_transform(X3)
X6=CountVectorizer(analyzer=t6,vocabulary=VOC6,dtype=np.float32).transform(UNIVERSE)
assert X3.shape==(len(UNIVERSE),64) and X6.shape==(len(UNIVERSE),3840)
print('Encoded',X3.shape,X6.shape,'nnz',X6.nnz)
def feature_rows(seq):return np.fromiter((INDEX[s] for s in seq),dtype=int,count=len(seq))
def sample_cohort(stage,cohort,sham_seed=None):
 d=FRAME[stage];train=d[d.original_split=='train'].copy()
 test_ids=set(d.loc[d.original_split=='test','record_id'])
 if cohort=='remove_exact_test_matches':train=train[~train.record_id.isin(test_ids)]
 elif cohort=='remove_related_test_groups':train=train[train.record_id.isin(
  set(manifest[stage].loc[manifest[stage].role=='train_candidate','record_id']))]
 elif cohort=='sham_positive_removal':
  leaking=set(train.record_id)&test_ids
  candidates=train[(train.label==1)&(~train.record_id.isin(leaking))].copy()
  targets=train[train.record_id.isin(leaking)].copy()
  rng=np.random.default_rng(sham_seed);available=candidates.copy();chosen=[]
  for target in targets.itertuples():
   gc_target=target.sequence.count('G')+target.sequence.count('C')
   dist=available.sequence.map(lambda s:abs(s.count('G')+s.count('C')-gc_target)).to_numpy()
   near=np.flatnonzero(dist==dist.min());ix=int(rng.choice(near))
   chosen.append(available.iloc[ix].record_id)
   available=available.drop(available.index[ix])
  train=train[~train.record_id.isin(chosen)]
 else:assert cohort=='historical_raw'
 assert train.record_id.is_unique
 return train.reset_index(drop=True)
def predict_baseline(stage,cohort,model,sham_seed=None):
 tr=sample_cohort(stage,cohort,sham_seed)
 te=FRAME[stage].query("original_split == 'test'").reset_index(drop=True)
 ids=feature_rows(tr.sequence.tolist()+[rc(s) for s in tr.sequence])
 tid=feature_rows(te.sequence)
 y=np.tile(tr.label.to_numpy(dtype=int),2)
 if model=='gc_cpg_logreg':
  clf=make_pipeline(StandardScaler(),LogisticRegression(C=1,max_iter=600,random_state=20261015))
  a,b=GC[ids],GC[tid]
 elif model=='k3_logreg':
  clf=LogisticRegression(C=1,max_iter=600,random_state=20261015)
  a,b=X3[ids],X3[tid]
 elif model=='spaced6_4_logreg':
  tf=TfidfTransformer(norm='l2').fit(X6[ids]);a,b=tf.transform(X6[ids]),tf.transform(X6[tid])
  clf=LogisticRegression(C=1,max_iter=600,solver='liblinear',random_state=20261015)
 else:raise ValueError(model)
 clf.fit(a,y)
 p=clf.predict_proba(b)[:,1]
 assert np.isfinite(p).all() and ((p>=0)&(p<=1)).all()
 return pd.DataFrame({'stage':stage,'cohort':cohort,'model':model,'sham_seed':sham_seed,
  'record_id':te.record_id,'label':te.label,'score':p,
  'exact_training_match':te.record_id.isin(set(FRAME[stage].query("original_split == 'train'").record_id)),
  'training_rows':len(tr)})

In [ ]:
# 4 — Execute 18 planned fits plus 10 GC-matched sham fits. Individual prediction files resume safely.
MODELS=CONFIG['models'];COHORTS=CONFIG['training_cohorts']
def cached_fit(stage,cohort,model,sham_seed=None):
 key=f's{stage}_{cohort}_{model}'+(f'_sham{sham_seed}' if sham_seed is not None else '')
 file=CACHE/f'{key}.csv';meta=CACHE/f'{key}.json'
 if file.exists() and meta.exists():
  m=json.loads(meta.read_text())
  if m.get('protocol')==LOCK and m.get('sha256')==sha(file.read_bytes()):
   df=pd.read_csv(file,dtype={'record_id':str,'cohort':str,'model':str})
   expected=FRAME[stage].query("original_split == 'test'").record_id.tolist()
   assert df.record_id.tolist()==expected and len(df)==len(expected)
   print('REUSED',key,flush=True);return df
 df=predict_baseline(stage,cohort,model,sham_seed)
 tmp=file.with_suffix('.part');df.to_csv(tmp,index=False);os.replace(tmp,file)
 meta.write_text(json.dumps({'protocol':LOCK,'sha256':sha(file.read_bytes())},indent=2))
 print('FITTED',key,'n_train',int(df.training_rows.iloc[0]),flush=True)
 return df
parts=[]
for s in (1,2):
 for cohort in COHORTS:
  for model in MODELS:parts.append(cached_fit(s,cohort,model))
 for rep in range(CONFIG['sham_repeats']):
  parts.append(cached_fit(s,'sham_positive_removal','spaced6_4_logreg',
                          CONFIG['seed']+100*s+rep))
pred=pd.concat(parts,ignore_index=True)
assert not pred.duplicated(['stage','cohort','model','sham_seed','record_id']).any()
pred.to_csv(WORK/'historical_test_predictions.csv',index=False)
print('Prediction rows:',len(pred))

In [ ]:
# 5 — Descriptive metrics; undefined discrimination is explicit for one-class Stage 2.
def measure(g,subset):
 y=g.label.to_numpy(dtype=int);score=g.score.to_numpy(dtype=float)
 out={'stage':int(g.stage.iloc[0]),'cohort':g.cohort.iloc[0],
      'model':g.model.iloc[0], 'sham_seed':g.sham_seed.iloc[0],
      'subset':subset,'n':len(g),'positive':int(y.sum()),'negative':int((y==0).sum()),
      'mean_positive_score':float(score[y==1].mean()) if (y==1).any() else np.nan,
      'mean_negative_score':float(score[y==0].mean()) if (y==0).any() else np.nan,
      'sensitivity_at_0_5':float((score[y==1]>=.5).mean()) if (y==1).any() else np.nan,
      'specificity_at_0_5':float((score[y==0]<.5).mean()) if (y==0).any() else np.nan,
      'brier':float(brier_score_loss(y,score)),
      'undefined_reason':'' if len(np.unique(y))==2 else 'one label class after test removal'}
 out['auroc']=float(roc_auc_score(y,score)) if len(np.unique(y))==2 else np.nan
 out['auprc']=float(average_precision_score(y,score)) if (y==1).any() else np.nan
 return out
rows=[]
for (_,_,_,_),g in pred.groupby(['stage','cohort','model','sham_seed'],dropna=False,sort=False):
 rows.append(measure(g,'full_historical_test'))
 clean=g[~g.exact_training_match]
 rows.append(measure(clean,'test_without_exact_matches'))
metrics=pd.DataFrame(rows)
metrics.to_csv(WORK/'historical_test_metrics.csv',index=False)
assert metrics.query("stage == 2 and subset == 'test_without_exact_matches'").auroc.isna().all()
print('Main cohort AUROC; Stage 2 clean subset intentionally NA:')
print(metrics[(metrics.cohort!='sham_positive_removal')][['stage','cohort','model','subset','n','positive','auroc','auprc']].to_string(index=False))

In [ ]:
# 6 — Paired stratified test bootstrap, holding test membership fixed for each contrast.
# This quantifies uncertainty conditional on the supplied test, not a claim of new independent validation.
boot=[];rng=np.random.default_rng(CONFIG['seed'])
for s in (1,2):
 for model in MODELS:
  block=pred[(pred.stage==s)&(pred.model==model)&(pred.cohort.isin(COHORTS))]
  wide=block.pivot(index='record_id',columns='cohort',values='score')
  truth=FRAME[s].query("original_split == 'test'").set_index('record_id').loc[wide.index]
  for subset in ('full_historical_test','test_without_exact_matches'):
   use=np.ones(len(wide),dtype=bool) if subset=='full_historical_test' else ~truth.index.isin(
    set(FRAME[s].query("original_split == 'train'").record_id))
   yy=truth.label.to_numpy(dtype=int)[use]; ww=wide.iloc[np.flatnonzero(use)]
   if len(np.unique(yy))!=2:continue
   pos=np.flatnonzero(yy==1);neg=np.flatnonzero(yy==0)
   comparisons=[('historical_raw','remove_exact_test_matches'),
                ('historical_raw','remove_related_test_groups')]
   for a,b in comparisons:
    pa=ww[a].to_numpy();pb=ww[b].to_numpy()
    obs=roc_auc_score(yy,pa)-roc_auc_score(yy,pb)
    deltas=np.empty(CONFIG['bootstrap_repeats'])
    for i in range(len(deltas)):
     ix=np.concatenate([rng.choice(pos,len(pos),replace=True),rng.choice(neg,len(neg),replace=True)])
     deltas[i]=roc_auc_score(yy[ix],pa[ix])-roc_auc_score(yy[ix],pb[ix])
    lo,hi=np.quantile(deltas,[.025,.975])
    boot.append({'stage':s,'model':model,'subset':subset,'contrast':a+' minus '+b,
                 'delta_auroc':obs,'lower_95':lo,'upper_95':hi,
                 'resampling_unit':'test record; stratified by class',
                 'bootstrap_repeats':len(deltas)})
bootstrap=pd.DataFrame(boot);bootstrap.to_csv(WORK/'paired_test_bootstrap.csv',index=False)
print(bootstrap.to_string(index=False))
# Compare against prespecified positive GC-matched row-removal controls (not a randomization p value).
shams=metrics[(metrics.cohort=='sham_positive_removal')&
              (metrics.subset=='full_historical_test')][['stage','sham_seed','auroc']]
shams.to_csv(WORK/'sham_removal_auroc.csv',index=False)
print('Sham controls:\n',shams.to_string(index=False))

In [ ]:
# 7 — Frozen grouped-CV context; its training/test structure differs from the historical test.
selected=context[context.model.isin(['spaced6_4_logreg','finetuned_dnabert_legacy',
                                     'finetuned_dnabert2'])].copy()
assert len(selected)==6
selected.to_csv(WORK/'phase2_grouped_cv_context.csv',index=False)
print('Separate Phase 2.1 grouped CV context:')
print(selected[['model','stage','mean_fold_auroc','sd_fold_auroc','pooled_oof_auroc']].to_string(index=False))
print('Do not subtract these CV values from historical-test AUROCs as a causal estimate.')

In [ ]:
# 8 — Publication-oriented descriptive figures, then a ZIP with member SHA256 provenance.
main=metrics[(metrics.cohort!='sham_positive_removal') & (metrics.stage==1)].copy()
fig,ax=plt.subplots(figsize=(11,5))
x=np.arange(len(MODELS));w=.12
for j,cohort in enumerate(COHORTS):
 for k,subset in enumerate(('full_historical_test','test_without_exact_matches')):
  v=[main[(main.model==m)&(main.cohort==cohort)&(main.subset==subset)].auroc.iloc[0] for m in MODELS]
  ax.bar(x+(2*j+k-2.5)*w,v,width=w,label=cohort+(' / full' if k==0 else ' / nonduplicate'))
ax.set_xticks(x,MODELS);ax.set_ylim(0,1);ax.set_ylabel('AUROC on supplied Stage 1 historical test')
ax.legend(fontsize=7,ncol=2,loc='lower right');ax.grid(axis='y',alpha=.2)
fig.tight_layout();fig.savefig(WORK/'stage1_historical_overlap_auroc.png',dpi=190);fig.savefig(WORK/'stage1_historical_overlap_auroc.svg');plt.close(fig)
fig,ax=plt.subplots(figsize=(8,4))
counts=roster.groupby(['stage','exact_training_match','label']).size()
for i,s in enumerate((1,2)):
 for j,matched in enumerate((False,True)):
  b=i*3+j;neg=int(counts.get((s,matched,0),0));pos=int(counts.get((s,matched,1),0))
  ax.bar(b,neg,color='#5470aa',label='negative' if b==0 else None)
  ax.bar(b,pos,bottom=neg,color='#e38437',label='positive' if b==0 else None)
ax.set_xticks([0,1,3,4],['S1 nonmatch','S1 match','S2 nonmatch','S2 match']);ax.set_ylabel('Test records')
ax.legend();fig.tight_layout();fig.savefig(WORK/'historical_test_overlap_composition.png',dpi=190)
fig.savefig(WORK/'historical_test_overlap_composition.svg');plt.close(fig)
readme=("Phase 11A: audit of supplied historical independent files and refit of three CPU baselines.\n"
"All cleaned Stage 2 test outcomes are negative; AUROC and AUPRC on that subset are undefined.\n"
"Historical test is secondary and previously examined. No causal difference between grouped CV and historical holdout is claimed.\n"
"Rows removed from training and rows removed from test are different comparisons.\n")
(WORK/'README_INTERPRETATION.txt').write_text(readme)
files=['historical_test_overlap_roster.csv','historical_test_predictions.csv',
 'historical_test_metrics.csv','paired_test_bootstrap.csv','sham_removal_auroc.csv',
 'phase2_grouped_cv_context.csv','stage1_historical_overlap_auroc.png',
 'stage1_historical_overlap_auroc.svg','historical_test_overlap_composition.png',
 'historical_test_overlap_composition.svg','README_INTERPRETATION.txt']
provenance={'protocol_sha256':LOCK,'config':CONFIG,
 'stage1_manifest_sha256':HASH['stage1_candidate_manifest.csv'],
 'stage2_manifest_sha256':HASH['stage2_candidate_manifest.csv'],
 'phase2_comparison_sha256':HASH['comparison_mean_fold_vs_pooled.csv'],
 'versions':{'python':platform.python_version(),'numpy':np.__version__,
             'pandas':pd.__version__,'scipy':scipy.__version__,'scikit_learn':sklearn.__version__},
 'output_sha256':{n:sha((WORK/n).read_bytes()) for n in files},
 'limitations':summary['limitations']+[
  'The Phase 1 LCS grouping status is a candidate sequence protocol requiring review.',
  'Five GC-matched deletion shams are descriptive, not a full causal randomization test.',
  'Historical tests were previously examined; confidence intervals condition on these files.',
  'No cleaned Stage 2 discrimination metric can be computed because the retained set has no positive records.']}
(WORK/'phase11a_provenance.json').write_text(json.dumps(provenance,indent=2,sort_keys=True)+'\n')
archive=WORK.parent/'enhancer_phase11a_overlap_impact.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
 for n in files+['phase11a_provenance.json']:z.write(WORK/n,n)
with zipfile.ZipFile(archive) as z:
 assert z.testzip() is None
 for n,h in provenance['output_sha256'].items():assert sha(z.read(n))==h
print('DONE:',archive,'bytes',archive.stat().st_size,'SHA256',sha(archive.read_bytes()))